In [1]:
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# -----------------------------------------------------------------------------
# Data loading (same as other notebooks)
# -----------------------------------------------------------------------------
csv_path = '../../robotic_arm_dataset_multiple_trajectories.csv'
df = pd.read_csv(csv_path)
data = torch.from_numpy(df[['Axis_0_Angle', 'Axis_1_Angle', 'Axis_2_Angle']].values.astype(np.float32))
T, d = data.shape
print(f'Loaded ONE long trajectory: T={T}, d={d}')

def make_splits(series, train_frac=0.7, val_frac=0.15):
    T = len(series)
    train_end = int(train_frac * T)
    val_end = int((train_frac + val_frac) * T)
    train = series[:train_end]
    val = series[train_end:val_end]
    test = series[val_end:]
    return train, val, test

train_series, val_series, test_series = make_splits(data)


class SlidingWindowDataset(Dataset):
    def __init__(self, series, K):
        self.series = series
        self.K = K

    def __len__(self):
        return len(self.series) - self.K

    def __getitem__(self, idx):
        x = self.series[idx:idx + self.K]
        y = self.series[idx + self.K]
        return x, y


class RNNRegressor(nn.Module):
    def __init__(self, d_in, hidden_size):
        super().__init__()
        self.rnn = nn.GRU(d_in, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, d_in)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        n += batch_size
    return total_loss / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        loss = criterion(pred, y)
        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        n += batch_size
    return total_loss / n


import copy

def evaluate_with_ttt(base_model, test_series, K, device, adapt_lr=1e-4, adapt_steps=1):
    """One-step-ahead TTT along the test trajectory, returning MSE & MAE."""
    model = copy.deepcopy(base_model).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=adapt_lr)
    criterion_mse = nn.MSELoss()
    model.train()
    T_test = len(test_series)
    preds = []
    targets = []
    for start in range(T_test - K):
        window = test_series[start:start + K].unsqueeze(0).to(device)
        target = test_series[start + K].unsqueeze(0).to(device)
        for _ in range(adapt_steps):
            optimizer.zero_grad()
            pred = model(window)
            loss = criterion_mse(pred, target)
            loss.backward()
            optimizer.step()
        with torch.no_grad():
            pred = model(window)
        preds.append(pred.squeeze(0).cpu().numpy())
        targets.append(target.squeeze(0).cpu().numpy())
    preds = np.stack(preds)
    targets = np.stack(targets)
    mse = ((preds - targets) ** 2).mean()
    mae = np.abs(preds - targets).mean()
    return preds, targets, float(mse), float(mae)


# -----------------------------------------------------------------------------
# Grid over H in {1..10} and K in {10, 50, 100}
# -----------------------------------------------------------------------------
Ks = [10, 50, 100]
H_list = list(range(1, 11))
num_epochs = 20
batch_size = 256
criterion_mse = nn.MSELoss()
criterion_mae = nn.L1Loss()

# results[(K, H)] = dict of metrics
results = {}

for K in Ks:
    print('\n==============================')
    print(f'K = {K}')
    print('==============================')
    train_ds = SlidingWindowDataset(train_series, K)
    val_ds = SlidingWindowDataset(val_series, K)
    test_ds = SlidingWindowDataset(test_series, K)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    base_test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    for H in H_list:
        model = RNNRegressor(d_in=d, hidden_size=H).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        best_val_mse = float('inf')
        best_state = None

        for epoch in range(1, num_epochs + 1):
            train_mse = train_one_epoch(model, train_loader, optimizer, criterion_mse, device)
            val_mse = eval_epoch(model, val_loader, criterion_mse, device)
            val_mae = eval_epoch(model, val_loader, criterion_mae, device)
            if val_mse < best_val_mse:
                best_val_mse = val_mse
                best_state = copy.deepcopy(model.state_dict())
            print(f'TTT grid | K={K} | H={H} | Epoch {epoch:02d} | train_MSE={train_mse:.6f} | val_MSE={val_mse:.6f} | val_MAE={val_mae:.6f} | best_val_MSE={best_val_mse:.6f}')

        # restore best model for this (K, H)
        model.load_state_dict(best_state)

        # base test metrics (no TTT)
        base_test_mse = eval_epoch(model, base_test_loader, criterion_mse, device)
        base_test_mae = eval_epoch(model, base_test_loader, criterion_mae, device)

        # TTT test metrics (one-step-ahead TTT along full test trajectory)
        _, _, ttt_mse, ttt_mae = evaluate_with_ttt(model, test_series, K, device, adapt_lr=1e-4, adapt_steps=1)

        results[(K, H)] = {
            'val_best_mse': best_val_mse,
            'base_test_mse': base_test_mse,
            'base_test_mae': base_test_mae,
            'ttt_mse': ttt_mse,
            'ttt_mae': ttt_mae,
        }

        print(f'>> (K={K}, H={H}) | val_best_MSE={best_val_mse:.6f} | base test MSE={base_test_mse:.6f} | base test MAE={base_test_mae:.6f} | TTT test MSE={ttt_mse:.6f} | TTT test MAE={ttt_mae:.6f}')

# -----------------------------------------------------------------------------
# Summarize all 30 configurations in a table (for easy copy into report)
# -----------------------------------------------------------------------------
rows = []
for (K, H), m in results.items():
    rows.append({
        'K': K,
        'H': H,
        'val_best_MSE': m['val_best_mse'],
        'base_test_MSE': m['base_test_mse'],
        'base_test_MAE': m['base_test_mae'],
        'TTT_test_MSE': m['ttt_mse'],
        'TTT_test_MAE': m['ttt_mae'],
    })
results_df = pd.DataFrame(rows).sort_values(['K', 'H']).reset_index(drop=True)
print('\n===== Full grid results (30 configs) =====')
print(results_df)

# Optionally: one quick plot example, e.g., base vs TTT MSE over H for each K
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for i, K in enumerate(Ks):
    ax = axes[i]
    base_mse = [results[(K, H)]['base_test_mse'] for H in H_list]
    ttt_mse = [results[(K, H)]['ttt_mse'] for H in H_list]
    ax.plot(H_list, base_mse, 'o-', label='base test MSE')
    ax.plot(H_list, ttt_mse, 's-', label='TTT test MSE')
    ax.set_title(f'K={K}')
    ax.set_xlabel('H')
axes[0].set_ylabel('MSE')
axes[0].legend()
plt.tight_layout()
plt.show()


Using device: cpu
Loaded ONE long trajectory: T=152500, d=3

K = 10
TTT grid | K=10 | H=1 | Epoch 01 | train_MSE=24.982844 | val_MSE=31.600106 | val_MAE=4.530499 | best_val_MSE=31.600106
TTT grid | K=10 | H=1 | Epoch 02 | train_MSE=21.575524 | val_MSE=27.679589 | val_MAE=4.194259 | best_val_MSE=27.679589
TTT grid | K=10 | H=1 | Epoch 03 | train_MSE=18.995928 | val_MSE=25.220809 | val_MAE=4.046707 | best_val_MSE=25.220809
TTT grid | K=10 | H=1 | Epoch 04 | train_MSE=17.183414 | val_MSE=23.199843 | val_MAE=3.897941 | best_val_MSE=23.199843
TTT grid | K=10 | H=1 | Epoch 05 | train_MSE=15.690318 | val_MSE=21.470618 | val_MAE=3.761655 | best_val_MSE=21.470618
TTT grid | K=10 | H=1 | Epoch 06 | train_MSE=14.469893 | val_MSE=19.990358 | val_MAE=3.640499 | best_val_MSE=19.990358
TTT grid | K=10 | H=1 | Epoch 07 | train_MSE=13.453251 | val_MSE=18.711428 | val_MAE=3.535279 | best_val_MSE=18.711428
TTT grid | K=10 | H=1 | Epoch 08 | train_MSE=12.600502 | val_MSE=17.601568 | val_MAE=3.441714 | bes

KeyboardInterrupt: 